# **AU2544013 - Hardi Makwana**
# **AU2544030 - Krisha Doshi**

# **Data Cleaning Pipeline**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import math
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_absolute_error

# **Loading Dataset**

In [ ]:
# Load the dataset
df = pd.read_csv('/content/drive/MyDrive/Data Science Project/MSME_Credit_Data_30S_CR.csv')

In [ ]:
# Basic info to identify numeric and categorical columns
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
print(df.info())

In [ ]:
df.describe()

# **Step 1: Identify and Preserve Critical Columns**

In [ ]:
# Ensure unique identifiers and labels are not altered
critical_cols = ['Enterprise_id', 'Label']
preserved_data = df[critical_cols].copy()

# Identify all numerical and categorical features for cleaning
numeric_features = df.select_dtypes(include=[np.number]).columns.drop(critical_cols, errors='ignore').tolist()
categorical_features = df.select_dtypes(exclude=[np.number]).columns.tolist()

print(f"Data Loaded. Numeric features: {len(numeric_features)}, Categorical: {len(categorical_features)}")
print(f"\nLabel distribution:")
print(df['Label'].value_counts())
print(f"\nMissing values (columns with missing only):")
missing = df.isnull().sum()
print(missing[missing > 0])

# **Step 2: Log Transformation (Replacing Outlier Treatment)**

### Why NO outlier removal for this dataset?
This is MSME credit risk data. Most columns are **event/count columns** (penalties, trademarks, legal cases etc.) where:
- `0` means the event never happened — this is meaningful, not an outlier
- High values mean the event happened many times — also meaningful, not an error
- Removing or replacing these values with median would destroy the most important credit risk signals

### What we do instead: Log Transform skewed continuous columns
For columns that are genuinely continuous and heavily right-skewed (capital, employee counts etc.),
we apply `log1p` which compresses large values without removing them.
`log1p(x) = log(x + 1)` — safe for zeros too.

In [ ]:
# Columns to apply log transformation:
# These are skewed continuous columns where variation is real but scale is large
log_transform_cols = [
    'Registered_capital (Ten thousand Yuan)',
    'Paid_in_capital (Ten thousand Yuan)',
    'SH_num',
    'MS_num',
    'Branch_num',
    'CL_5years+', 'CL_5years', 'CL_4years', 'CL_2years', 'CL_1year',
    'Certificate_num_2years', 'Certificate_num_1year',
    't-1 Basic old-age insurance for urban employees',
    't-2 Basic old-age insurance for urban employees',
    't-3 Basic old-age insurance for urban employees',
    't-1 Unemployment insurance',
    't-2 Unemployment insurance',
    't-3 Unemployment insurance',
    't-1 Basic medical insurance for employees',
    't-2 Basic medical insurance for employees',
    't-3 Basic medical insurance for employees',
    't-1 Employment injury insurance',
    't-2 Employment injury insurance',
    't-3 Employment injury insurance',
    't-1 Birth insurance',
    't-2 Birth insurance',
    't-3 Birth insurance',
    'Legal_proceedings_num_2years',
    'Legal_proceedings_num_1year',
    'Filing_information_num_1years'
]

# Only transform columns that actually exist in the dataset
log_transform_cols = [c for c in log_transform_cols if c in df.columns]

df_treated = df.copy()

for col in log_transform_cols:
    # log1p handles 0s safely: log(0+1) = 0, so zeros remain 0
    df_treated[col] = np.log1p(df_treated[col])

print(f"Log transformation applied to {len(log_transform_cols)} columns.")
print("All zero-inflated event/count columns left untouched.")
print("\nColumns NOT transformed (left as-is):")
not_transformed = [c for c in numeric_features if c not in log_transform_cols]
print(f"  {len(not_transformed)} columns — includes all penalty, trademark, patent, legal count columns")

# **Step 3: Missing Value Imputation**

### Missing value summary:
- `Branch_num`: 44.6% missing — large number, but KNN/MICE can handle it
- Insurance columns (t-2, t-3): ~3-8% missing — likely companies that didn't report
- `MS_num`, `SH_num`, `Establishment_Duration`: very few missing (<1%)

In [ ]:
# Create Evaluation Set from ALL completely non-null rows
eval_set = df_treated.dropna().copy()
true_values = eval_set[numeric_features].copy()

print(f"Rows with no missing values (used for evaluation): {len(eval_set)}")

# Introduce 10% Artificial Missingness across ALL cells of numerical columns
eval_df_masked = eval_set.copy()
mask = np.random.RandomState(42).rand(*eval_df_masked[numeric_features].shape) < 0.1
eval_df_masked[numeric_features] = eval_df_masked[numeric_features].mask(mask)

print(f"Artificial missingness introduced. Evaluating imputation methods...")

# **Step 4: Compare Imputation Methods**

In [ ]:
# Define Imputers to compare
imputers = {
    'Mean': SimpleImputer(strategy='mean'),
    'KNN': KNNImputer(n_neighbors=5),
    'MICE': IterativeImputer(max_iter=10, random_state=0)
}

results = []

In [ ]:
# Perform Imputation and Calculate Metrics for ALL columns
for name, imputer in imputers.items():
    imputed_array = imputer.fit_transform(eval_df_masked[numeric_features])
    imputed_df = pd.DataFrame(imputed_array, columns=numeric_features, index=eval_set.index)

    for col in numeric_features:
        col_mask = mask[:, numeric_features.index(col)]
        if col_mask.any():
            y_true = true_values.loc[col_mask, col]
            y_pred = imputed_df.loc[col_mask, col]

            mae = mean_absolute_error(y_true, y_pred)
            raw_bias = (y_pred - y_true).mean()
            mean_val = y_true.mean()
            pct_bias = (raw_bias / mean_val * 100) if mean_val != 0 else 0

            results.append({
                'Method': name,
                'Column': col,
                'MAE': mae,
                'Raw Bias': raw_bias,
                'Percentage Bias': pct_bias
            })

perf_df = pd.DataFrame(results)
print("Imputation evaluation complete!")
print("\nAverage MAE per method:")
print(perf_df.groupby('Method')['MAE'].mean().sort_values())

In [ ]:
# Visualize Imputation Performance
metrics = ['MAE', 'Raw Bias', 'Percentage Bias']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, metric in enumerate(metrics):
    sns.barplot(
        data=perf_df,
        x='Method',
        y=metric,
        ax=axes[i],
        palette='coolwarm',
        errorbar=None,
        hue='Method',
        legend=False
    )
    axes[i].set_title(f'Average {metric} Across Columns')

plt.tight_layout()
plt.show()

# **Step 5: Apply Best Imputation Method & Save**

In [ ]:
# Select best method based on lowest MAE
best_method_name = perf_df.groupby('Method')['MAE'].mean().idxmin()
print(f"Best performing method: {best_method_name}")

# Apply best imputer to the FULL log-transformed dataset
final_imputer = imputers[best_method_name]
features_df = df_treated.drop(columns=critical_cols)

# Impute numeric features
cleaned_numeric = pd.DataFrame(
    final_imputer.fit_transform(features_df[numeric_features]),
    columns=numeric_features
)

# Impute categorical features using Mode
cat_imputer = SimpleImputer(strategy='most_frequent')
cleaned_categorical = pd.DataFrame(
    cat_imputer.fit_transform(features_df[categorical_features]),
    columns=categorical_features
)

# Final Recombination
df_final = pd.concat([preserved_data.reset_index(drop=True), cleaned_numeric, cleaned_categorical], axis=1)

print(f"Final cleaned dataset shape: {df_final.shape}")
print(f"Any remaining missing values: {df_final.isnull().sum().sum()}")

# Save
df_final.to_csv('/content/drive/MyDrive/Data Science Project/MSME_Credit_Cleaned_Final.csv', index=False)
print("Saved to: MSME_Credit_Cleaned_Final.csv")

# **EDA — Exploratory Data Analysis**

In [ ]:
# Reload the cleaned file (or use df_final directly)
# df_final = pd.read_csv('/content/drive/MyDrive/Data Science Project/MSME_Credit_Cleaned_Final.csv')

print(f"Dataset Shape: {df_final.shape}")
print(f"\nLabel Distribution:")
print(df_final['Label'].value_counts())
print(f"\nDefault Rate: {df_final['Label'].mean()*100:.2f}%")

sns.set(style="whitegrid")
num_features_eda = df_final.select_dtypes(include=[np.number]).columns.drop(['Enterprise_id', 'Label'], errors='ignore').tolist()

## EDA 1: Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
sns.countplot(data=df_final, x='Label', hue='Label', palette='Set1', legend=False, ax=axes[0])
axes[0].set_title('Target Distribution (0: Non-Default, 1: Default)', fontsize=13)
axes[0].set_xlabel('Label')
axes[0].set_ylabel('Count')
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom', fontsize=12)

# Pie chart
label_counts = df_final['Label'].value_counts()
axes[1].pie(label_counts, labels=['Non-Default (0)', 'Default (1)'],
            autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'], startangle=90)
axes[1].set_title('Default vs Non-Default Proportion', fontsize=13)

plt.tight_layout()
plt.show()
print("Note: Dataset is highly imbalanced — 94% Non-Default vs 6% Default")

## EDA 2: Categorical Feature Distributions

In [ ]:
cat_features = df_final.select_dtypes(exclude=[np.number]).columns

for col in cat_features:
    plt.figure(figsize=(12, 5))
    order = df_final[col].value_counts().index
    sns.countplot(data=df_final, x=col, hue=col, palette='viridis', legend=False, order=order)
    plt.title(f'Distribution of {col}', fontsize=14)
    plt.xlabel(col, fontsize=11)
    plt.ylabel('Count', fontsize=11)
    plt.xticks(rotation=45, ha='right', fontsize=9)
    plt.tight_layout()
    plt.show()
    print("\n")

## EDA 3: Numerical Feature Distributions (Histograms)

In [ ]:
# Show distributions of first 12 numerical features
plt.figure(figsize=(20, 16))
for i, col in enumerate(num_features_eda[:12]):
    plt.subplot(4, 3, i+1)
    sns.histplot(df_final[col], kde=True, color='teal')
    plt.title(f'{col}', fontsize=9)
    plt.xlabel('')
plt.suptitle('Distribution of First 12 Numeric Features (After Log Transform)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## EDA 4: Numerical Features vs Label (Box Plots)

In [ ]:
# Box plots — key features vs Default/Non-Default
key_features = [
    'Registered_capital (Ten thousand Yuan)',
    'Establishment_Duration (Days)',
    'Paid_in_capital (Ten thousand Yuan)',
    'SH_num', 'MS_num', 'Branch_num',
    'CL_1year', 'CL_2years', 'CL_3years'
]
key_features = [c for c in key_features if c in df_final.columns]

plt.figure(figsize=(20, 15))
for i, col in enumerate(key_features):
    plt.subplot(3, 3, i+1)
    sns.boxplot(data=df_final, x='Label', y=col, hue='Label', palette='Set2', legend=False)
    plt.title(f'{col} vs Label', fontsize=10)
    plt.xlabel('Label (0=Non-Default, 1=Default)')
plt.tight_layout()
plt.show()

## EDA 5: Correlation Heatmap

In [ ]:
# Drop zero-variance columns to avoid white squares
data_for_heatmap = df_final[num_features_eda + ['Label']].copy()
data_for_heatmap = data_for_heatmap.loc[:, data_for_heatmap.std() > 0]

plt.figure(figsize=(18, 14))
corr_matrix = data_for_heatmap.corr()
sns.heatmap(
    corr_matrix,
    annot=False,
    cmap='coolwarm',
    center=0,
    cbar_kws={'label': 'Correlation Coefficient'}
)
plt.title('MSME Credit Risk: Correlation Matrix (Cleaned Data)', fontsize=16)
plt.tight_layout()
plt.show()

# Show top correlations with Label
label_corr = corr_matrix['Label'].drop('Label').abs().sort_values(ascending=False)
print("Top 15 features correlated with Label (Default):")
print(label_corr.head(15))

## EDA 6: Default Rate by Sector (Stacked Bar)

In [ ]:
sector_label = pd.crosstab(df_final['Sector'], df_final['Label'])
sector_label_pct = sector_label.div(sector_label.sum(1), axis=0)

sector_label_pct.plot(kind='bar', stacked=True, figsize=(14, 7), color=['#4C72B0', '#DD8452'])
plt.title('Default Rate (Label) by Sector', fontsize=15)
plt.ylabel('Proportion', fontsize=12)
plt.xlabel('Sector', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(title='Label', labels=['Non-Default (0)', 'Default (1)'],
           bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## EDA 7: Compliance Modification Trend (Last 5 Years)

In [ ]:
# CL columns represent number of information modifications per year
# Check which CL cols are available
cl_cols = [c for c in ['CL_1year', 'CL_2years', 'CL_3years', 'CL_4years', 'CL_5years'] if c in df_final.columns]

trend_data = df_final.groupby('Label')[cl_cols].mean().T

plt.figure(figsize=(10, 5))
plt.plot(trend_data.index, trend_data[0], marker='o', label='Non-Default (0)', color='steelblue')
plt.plot(trend_data.index, trend_data[1], marker='s', label='Default (1)', color='tomato')
plt.title('Avg Information Modifications Trend (Last 5 Years) by Default Status', fontsize=13)
plt.ylabel('Mean Count (log-scaled)')
plt.xlabel('Year')
plt.legend()
plt.tight_layout()
plt.show()

## EDA 8: Registered Capital Quantile Distribution

In [ ]:
# Quantile Analysis — Registered Capital
df_final['Capital_Quantile'] = pd.qcut(
    df_final['Registered_capital (Ten thousand Yuan)'],
    q=4, labels=['Q1 (Smallest)', 'Q2', 'Q3', 'Q4 (Largest)']
)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart of distribution
df_final['Capital_Quantile'].value_counts().sort_index().plot.pie(
    autopct='%1.1f%%', colors=sns.color_palette('pastel'), startangle=140, ax=axes[0]
)
axes[0].set_title('Enterprise Distribution by Capital Quantile')
axes[0].set_ylabel('')

# Default rate by quantile
default_by_quantile = df_final.groupby('Capital_Quantile', observed=True)['Label'].mean() * 100
default_by_quantile.plot(kind='bar', ax=axes[1], color='salmon', edgecolor='black')
axes[1].set_title('Default Rate (%) by Capital Quantile')
axes[1].set_ylabel('Default Rate (%)')
axes[1].set_xlabel('Capital Quantile')
axes[1].tick_params(axis='x', rotation=15)
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%', (p.get_x() + p.get_width()/2., p.get_height()),
                     ha='center', va='bottom')

plt.tight_layout()
plt.show()

## EDA 9: Violin Plots — Density by Default Status

In [ ]:
selected_attr_violin = num_features_eda[:12]

num_plots = len(selected_attr_violin)
num_cols = 3
num_rows = math.ceil(num_plots / num_cols)

plt.figure(figsize=(24, 6 * num_rows))
for i, col in enumerate(selected_attr_violin):
    plt.subplot(num_rows, num_cols, i + 1)
    sns.violinplot(
        data=df_final,
        x='Label', y=col,
        hue='Label',
        inner="quart",
        palette="muted",
        legend=False
    )
    plt.title(f'{col} by Label', fontsize=11)
    plt.xlabel('Label (0=Non-Default, 1=Default)', fontsize=9)
    plt.ylabel(col, fontsize=9)

plt.tight_layout(pad=4.0)
plt.show()

## EDA 10: Regression Trend — Capital vs Enterprise Age

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(
    data=df_final,
    x='Registered_capital (Ten thousand Yuan)',
    y='Establishment_Duration (Days)',
    scatter_kws={'alpha': 0.2, 'color': 'steelblue'},
    line_kws={'color': 'red'}
)
plt.title('Relationship: Capital vs Enterprise Age (log-scaled axes)', fontsize=13)
plt.xlabel('Registered Capital (log-transformed)')
plt.ylabel('Establishment Duration (Days)')
plt.tight_layout()
plt.show()

## EDA 11: Insurance Coverage Trend by Default Status

In [ ]:
# Compare t-1, t-2, t-3 insurance coverage between default and non-default companies
insurance_types = [
    't-1 Basic old-age insurance for urban employees',
    't-2 Basic old-age insurance for urban employees',
    't-3 Basic old-age insurance for urban employees'
]
insurance_types = [c for c in insurance_types if c in df_final.columns]

ins_trend = df_final.groupby('Label')[insurance_types].mean().T
ins_trend.index = ['t-1 (Recent)', 't-2', 't-3 (Oldest)']

plt.figure(figsize=(9, 5))
plt.plot(ins_trend.index, ins_trend[0], marker='o', label='Non-Default (0)', color='steelblue')
plt.plot(ins_trend.index, ins_trend[1], marker='s', label='Default (1)', color='tomato')
plt.title('Avg Old-Age Insurance Coverage Trend (log-scaled) by Default Status', fontsize=12)
plt.ylabel('Mean Value (log-transformed employee count)')
plt.legend()
plt.tight_layout()
plt.show()

## EDA 12: Province-wise Default Rate

In [ ]:
province_default = df_final.groupby('Province')['Label'].agg(['mean', 'count']).reset_index()
province_default.columns = ['Province', 'Default_Rate', 'Count']
province_default['Default_Rate'] = province_default['Default_Rate'] * 100
province_default = province_default[province_default['Count'] >= 10]  # only provinces with enough data
province_default = province_default.sort_values('Default_Rate', ascending=False)

plt.figure(figsize=(14, 6))
bars = plt.bar(province_default['Province'], province_default['Default_Rate'],
               color='salmon', edgecolor='black')
plt.title('Default Rate (%) by Province (min 10 enterprises)', fontsize=13)
plt.ylabel('Default Rate (%)')
plt.xlabel('Province')
plt.xticks(rotation=45, ha='right')
# Add count labels
for bar, count in zip(bars, province_default['Count']):
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2,
             f'n={count}', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

## EDA 13: Event Columns — Default vs Non-Default (Penalty, Legal, Executee)

In [ ]:
# These are the zero-inflated columns we kept untouched
# Check if defaulted companies have more events
event_cols = [
    'Executee_num_1year', 'Overdue_tax_num_1year',
    'Administrative_penalty_num_1year', 'Legal_proceedings_num_1year'
]
event_cols = [c for c in event_cols if c in df_final.columns]

fig, axes = plt.subplots(1, len(event_cols), figsize=(16, 5))
for i, col in enumerate(event_cols):
    event_rate = df_final.groupby('Label')[col].apply(lambda x: (x > 0).mean() * 100)
    event_rate.plot(kind='bar', ax=axes[i], color=['#4C72B0', '#DD8452'], edgecolor='black')
    axes[i].set_title(f'% with {col} > 0', fontsize=9)
    axes[i].set_xlabel('Label')
    axes[i].set_ylabel('% of Companies')
    axes[i].tick_params(axis='x', rotation=0)
    for p in axes[i].patches:
        axes[i].annotate(f'{p.get_height():.1f}%',
                         (p.get_x() + p.get_width()/2., p.get_height()),
                         ha='center', va='bottom', fontsize=9)

plt.suptitle('Proportion of Companies with Recent Negative Events (0=Non-Default, 1=Default)',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()